[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/mongodb-partners/partner-cookbooks/blob/main/partners/langchain/claude-code-mcp/01_mongodb_mcp_and_agent.ipynb)

# MongoDB MCP Server + AI Agents

The **Model Context Protocol (MCP)** is an open standard that lets AI applications connect to
tools and data through a uniform interface. MongoDB ships an **MCP Server** that exposes your Atlas
cluster as a set of tools an agent can call (list collections, run queries and aggregations,
inspect schema, and more) without you writing a single custom tool.

This notebook connects a LangChain agent to MongoDB through the MCP Server and answers
questions about the same `sample_mflix` data from the Advanced RAG notebooks.

## What you'll learn
1. What the MongoDB MCP Server is and how to run it.
2. Load its tools into a LangChain agent with `langchain-mcp-adapters`.
3. Let Claude answer data questions by calling MongoDB tools.
4. Where this leads: the MongoDB plugin for Claude Code, and deploying agents.

## Prerequisites
- **Node.js 20+** (the MCP server runs on Node). Check with `node --version`.
- A MongoDB Atlas connection string and a Claude API key.
- The Advanced RAG cluster works well here, but any Atlas cluster with sample data is fine.


## Setup

In [1]:
%%capture
%pip install -q \
    langchain==1.3.12 langchain-anthropic==1.4.8 langchain-mcp-adapters==0.3.0 \
    langgraph==1.2.8 python-dotenv==1.2.2

# The MongoDB MCP Server is a Node package. Install it once:
# (@latest keeps the workshop current; pin a version if you need a fixed build.)
!npm install -g mongodb-mcp-server@latest

In [2]:
import os
from dotenv import load_dotenv
load_dotenv(override=True)

import getpass
def secret(name, prompt):
    v = os.getenv(name)
    return v if v and v.strip() else getpass.getpass(prompt)

MONGODB_URI   = secret("MONGODB_URI", "MongoDB Atlas connection string: ")
ANTHROPIC_KEY = secret("ANTHROPIC_API_KEY", "Claude API key: ")
ANTHROPIC_BASE_URL = os.getenv("ANTHROPIC_BASE_URL") or None
ANTHROPIC_MODEL    = os.getenv("ANTHROPIC_MODEL", "claude-haiku-4-5")
print("Configured.")

Configured.


## 1. What the MCP Server gives you

Run `mongodb-mcp-server` and it connects to your cluster and advertises a catalog of tools over
a standard protocol. Instead of hand-writing a "count documents" or "run aggregation" tool, the
agent discovers them automatically. We pass `--readOnly` so the agent can only read, a sensible
default when an LLM decides which tools to call.

We configure the LLM first. `configure_llm` supports both the Anthropic API and a compatible
gateway (set `ANTHROPIC_BASE_URL` for the latter; some gateways authenticate with an `api-key`
header, which we handle here).


In [3]:
from langchain_anthropic import ChatAnthropic

def configure_llm():
    kwargs = {"model": ANTHROPIC_MODEL, "max_tokens": 1024}
    if ANTHROPIC_BASE_URL:
        base = ANTHROPIC_BASE_URL
        for suffix in ("/v1/messages", "/messages"):
            if base.rstrip("/").endswith(suffix):
                base = base.rstrip("/")[: -len(suffix)]
                break
        kwargs.update(base_url=base, api_key="placeholder",
                      default_headers={"api-key": ANTHROPIC_KEY,
                                       "anthropic-version": "2023-06-01"})
    else:
        kwargs["api_key"] = ANTHROPIC_KEY
    return ChatAnthropic(**kwargs)

llm = configure_llm()
print("LLM ready:", ANTHROPIC_MODEL)

LLM ready: claude-haiku-4-5


## 2. Connect the agent to MongoDB over MCP

We start the MCP server as a subprocess (stdio transport), load its tools with
`langchain-mcp-adapters`, and hand them to a LangChain agent with `create_agent`. The agent decides which
MongoDB tools to call to answer the question.

> **Note on teardown noise:** on shutdown the async stdio transport can raise a harmless
> `BrokenResourceError` (sometimes wrapped in an `ExceptionGroup`) *after* the answer is already
> captured. We swallow only those specific teardown exceptions; real errors during the agent run
> still propagate so you see them.


In [4]:
from mcp import ClientSession, StdioServerParameters
from mcp.client.stdio import stdio_client
from langchain_mcp_adapters.tools import load_mcp_tools
from langchain.agents import create_agent

# Only these transport-teardown exceptions are treated as harmless noise.
try:
    from anyio import BrokenResourceError, ClosedResourceError
    _TEARDOWN_NOISE = (BrokenResourceError, ClosedResourceError)
except Exception:
    _TEARDOWN_NOISE = ()

async def ask_mongodb(question: str):
    params = StdioServerParameters(
        command="mongodb-mcp-server",
        args=[MONGODB_URI, "--readOnly"],
        env={"PATH": os.environ["PATH"], "MDB_MCP_TELEMETRY": "disabled"},
    )
    answer = None
    try:
        async with stdio_client(params) as (read, write):
            async with ClientSession(read, write) as session:
                await session.initialize()
                tools = await load_mcp_tools(session)
                print(f"Loaded {len(tools)} MongoDB tools, e.g.:",
                      ", ".join(t.name for t in tools[:6]))
                agent = create_agent(llm, tools)
                result = await agent.ainvoke({"messages": [{"role": "user", "content": question}]})
                answer = result["messages"][-1].content
    except* _TEARDOWN_NOISE:
        # Transport closed noisily after the answer was captured; ignore.
        pass
    return answer

# In a notebook the event loop is already running, so we can await directly.
answer = await ask_mongodb(
    "What collections are in the sample_mflix database, and how many documents "
    "are in the movies collection? Use the tools, then summarize."
)
print("\n" + (answer or "No answer returned."))

ANTHROPIC_API_KEY is set and takes precedence over the SDK's profile / federation auto-discovery; unset ANTHROPIC_API_KEY to use the auto-discovered credential.


Loaded 16 MongoDB tools, e.g.: aggregate-db, aggregate, collection-indexes, collection-schema, collection-storage-size, connect



## Summary

The **sample_mflix database** contains **38 collections**, including:

**Main Collections (Movie-related):**
- **movies** - containing **21,349 documents**
- **theaters**
- **users**
- **comments**
- **sessions**
- **embedded_movies**

**Mastra System Collections** (38 total, including the ones above):
The database also contains numerous Mastra framework system collections for features like:
- Workflow management (mastra_workflows, mastra_agents, mastra_skills, etc.)
- Experiment tracking (mastra_experiments, mastra_experiment_results)
- Data management (mastra_datasets, mastra_dataset_items)
- Prompt and AI management (mastra_prompt_blocks, mastra_ai_spans)
- And various other operational collections

The key finding is that the **movies collection has 21,349 documents**, making it one of the primary data collections in this sample database.


The agent listed the collections and counted the movies entirely through MCP tools, with no custom
tool code. Try another question that needs an aggregation.


In [5]:
answer = await ask_mongodb(
    "Using the sample_mflix.movies collection, what are the 5 most common movie genres? "
    "Use an aggregation, then give me the list with counts."
)
print(answer or "No answer returned.")

Loaded 16 MongoDB tools, e.g.: aggregate-db, aggregate, collection-indexes, collection-schema, collection-storage-size, connect


Perfect! Here are the **5 most common movie genres** in the sample_mflix.movies collection:

1. **Drama** - 12,385 movies
2. **Comedy** - 6,532 movies
3. **Romance** - 3,318 movies
4. **Crime** - 2,457 movies
5. **Thriller** - 2,454 movies

The aggregation pipeline I used:
1. **$unwind** - Deconstructed the genres array to create one document per genre per movie
2. **$group** - Grouped by genre and counted occurrences
3. **$sort** - Sorted by count in descending order
4. **$limit** - Limited the results to the top 5

Drama is by far the most common genre with nearly twice as many movies as Comedy!


## 3. Where this goes next

**MongoDB plugin for Claude Code.** The same MCP Server powers the MongoDB plugin for Claude Code,
so you can explore and query your cluster in natural language right in your terminal or IDE, no
notebook required.

**Agent Skills.** Skills package reusable expertise (schema design, query optimization, aggregation
patterns) that an agent loads on demand, complementing the MCP tools shown here.

**Deploying agents.** The agent you built runs locally today. The same tools-plus-LLM pattern
deploys to hosted agent platforms (for example, Claude's managed agents) when you're ready to move
from prototype to production.

## Recap

You ran the MongoDB MCP Server, loaded its tools into a LangChain agent, and let Claude answer
questions about `sample_mflix` by calling MongoDB directly, the foundation for connecting AI
assistants and developer tools to your data through one standard protocol.
